# BHaH Project Anatomy

Read the structure of a standalone BHaH project generated from the Cartesian wave equation.

Navigation: [Index](../index.ipynb) | Previous: [Multicoordinate Wave Project](../3-wave_equation/wave_equation_multicoordinates.ipynb) | Next: [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)


## Learning Goals

- Read a generated BHaH project as a set of roles, not as a pile of files.
- Connect parameters, prototypes, right-hand-side code, and diagnostics to earlier notebooks.
- Identify which files a beginner should inspect first.

## Words for This Notebook

- **BHaH:** BlackHoles@Home, the standalone C project style used by several NRPy examples.
- **Prototype header:** a file listing C functions so source files can call each other.
- **Source file:** a C file containing instructions to compile.
- **Diagnostic source:** C code that writes output used to check a run.

Use the code cells actively: first predict what should happen, then run the cell, then explain the output in plain language. This predict-run-explain pattern keeps the physics idea connected to the programming details.


## Generate a Standalone BHaH Project
This is the same real Cartesian wave project, read as an infrastructure artifact rather than as a first-run exercise.

## Import Project Inspection Helpers

These standard-library tools run commands, manage temporary project directories, and clean command output.

If you are new to Python, do not study this helper cell line by line on a first pass. Its job is practical: run terminal commands, shorten long command output, and stop with a clear message if a required tool is missing. Focus first on the cells that generate, inspect, build, run, and interpret the physics project.


In [1]:
from pathlib import Path
import re, shutil, subprocess, sys, tempfile


def clean_command_output(text):
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text or "")
    return cleaned.replace(str(WORKSPACE), "<workspace>")


def run_command(args, cwd, timeout):
    try:
        result = subprocess.run(
            args,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
            timeout=timeout,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required command is missing: {args[0]}") from exc
    except subprocess.CalledProcessError as exc:
        print(clean_command_output(exc.stdout))
        raise RuntimeError(f"Command failed: {' '.join(map(str, args))}") from exc
    return clean_command_output(result.stdout)


def require_toolchain():
    if shutil.which("make") is None:
        raise RuntimeError(
            "This notebook requires make to build the generated project."
        )
    if not any(shutil.which(name) for name in ["cc", "gcc", "clang"]):
        raise RuntimeError(
            "This notebook requires a C compiler such as cc, gcc, or clang."
        )


## Create an Anatomy Workspace

The workspace keeps generated files separate from the tutorial source tree.


In [2]:
PROJECT_NAME = "wave_equation_cartesian"
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_cartesian_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME


## Generate a Fresh Anatomy Project

This command invokes the same module a learner can run from a terminal and then verifies that the project directory exists.


In [3]:
command = [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"]
print("generator command: python -m nrpy.examples.wave_equation_cartesian")
output = run_command(command, WORKSPACE, timeout=300)
for line in output.splitlines():
    if line.strip():
        print(line.rstrip())
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)
print("project path: project/wave_equation_cartesian")


generator command: python -m nrpy.examples.wave_equation_cartesian


Finished! Now go into project/wave_equation_cartesian and type `make` to build, then ./wave_equation_cartesian to run.
    Parameter file can be found in wave_equation_cartesian.par
project path: project/wave_equation_cartesian


## Inventory and Core Files

In [4]:
required = [
    "Makefile",
    "BHaH_function_prototypes.h",
    "commondata_struct_set_to_default.c",
    "params_struct_set_to_default.c",
    "rhs_eval.c",
    "diagnostics.c",
    "wave_equation_cartesian.par",
]
for relative_path in required:
    path = PROJECT_DIR / relative_path
    if not path.is_file():
        raise FileNotFoundError(path)
print("top-level project entries:")
for path in sorted(PROJECT_DIR.iterdir()):
    print(path.name + ("/" if path.is_dir() else ""))


top-level project entries:
BHaH_defines.h
BHaH_function_prototypes.h
Makefile
MoL/
apply_bcs.c
cmdline_input_and_parfile_parser.c
commondata_struct_set_to_default.c
diagnostics.c
exact_solution_single_Cartesian_point.c
griddata_free.c
initial_data.c
intrinsics/
main.c
numerical_grids_and_timestep.c
params_struct_set_to_default.c
progress_indicator.c
rhs_eval.c
set_CodeParameters-nopointer.h
set_CodeParameters-simd.h
set_CodeParameters.h
wave_equation_cartesian.par


## Step 4: Read the Parameter File

This is the runtime file a learner edits before running the executable.

In [5]:
print(
    (PROJECT_DIR / "wave_equation_cartesian.par").read_text(
        encoding="utf-8", errors="replace"
    )
)


#### wave_equation_cartesian BH@H parameter file. NOTE: only commondata CodeParameters appear here ###
###########################
###########################
### Module: __main__
convergence_factor = 1.0        # (REAL)
diagnostics_output_every = 0.2  # (REAL)
###########################
###########################
### Module: nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData
sigma = 3.0                     # (REAL)
wavespeed = 1.0                 # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.MoLtimestepping.register_all
CFL_FACTOR = 0.5                # (REAL)
t_final = 8.0                   # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.diagnostics.progress_indicator
output_progress_every = 1       # (int)



## Step 5: Read the Prototype Header

The header shows the callable generated functions.

In [6]:
print(
    (PROJECT_DIR / "BHaH_function_prototypes.h").read_text(
        encoding="utf-8", errors="replace"
    )
)


void apply_bcs(const commondata_struct *restrict commondata, const params_struct *restrict params, REAL *restrict gfs);
void cmdline_input_and_parfile_parser(commondata_struct *restrict commondata, int argc, const char *argv[]);
void commondata_struct_set_to_default(commondata_struct *restrict commondata);
void diagnostics(commondata_struct *restrict commondata, griddata_struct *restrict griddata);
void exact_solution_single_Cartesian_point(const commondata_struct *restrict commondata, const params_struct *restrict params, const REAL xCart0,
                                           const REAL xCart1, const REAL xCart2, REAL *restrict exact_soln_UUGF, REAL *restrict exact_soln_VVGF);
void griddata_free(commondata_struct *restrict commondata, griddata_struct *restrict griddata,
                   const bool free_non_y_n_gfs_and_core_griddata_pointers);
void initial_data(const commondata_struct *restrict commondata, griddata_struct *restrict griddata);
int main(int argc, const char *arg

## Step 6: Read Default-Setting Sources

These files initialize generated runtime structures. The full files are worth reading here because they connect parameter names in the `.par` file to actual C assignments. As you read, look for the default values and the structure fields being filled.


In [7]:
for relative_path in [
    "commondata_struct_set_to_default.c",
    "params_struct_set_to_default.c",
]:
    print(f"--- {relative_path} ---")
    print((PROJECT_DIR / relative_path).read_text(encoding="utf-8", errors="replace"))


--- commondata_struct_set_to_default.c ---
#include "BHaH_defines.h"

/**
 * Set commondata_struct to default values specified within NRPy.
 */
void commondata_struct_set_to_default(commondata_struct *restrict commondata) {
  memset(commondata, 0, sizeof(*commondata));
  // Set commondata_struct variables to default
  commondata->CFL_FACTOR = 0.5;               // nrpy.infrastructures.BHaH.MoLtimestepping.register_all::CFL_FACTOR
  commondata->NUMGRIDS = 1;                   // nrpy.grid::NUMGRIDS
  commondata->convergence_factor = 1.0;       // __main__::convergence_factor
  commondata->diagnostics_output_every = 0.2; // __main__::diagnostics_output_every
  commondata->output_progress_every = 1;      // nrpy.infrastructures.BHaH.diagnostics.progress_indicator::output_progress_every
  commondata->sigma = 3.0;                    // nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData::sigma
  commondata->t_final = 8.0;                  // nrpy.infrastructures.BHaH.MoLtimestep

## Step 7: Read the Right-Hand-Side Source

The right-hand-side source contains the wave-equation update routine. The full file is printed because this is the anatomy notebook: look for the grid loop, the grid reads, the finite-difference expressions, and the assignments to the right-hand-side fields.


In [8]:
print((PROJECT_DIR / "rhs_eval.c").read_text(encoding="utf-8", errors="replace"))


#include "BHaH_defines.h"
#include "intrinsics/simd_intrinsics.h"

/**
 * Set RHSs for wave equation.
 */
void rhs_eval(const commondata_struct *restrict commondata, const params_struct *restrict params, const REAL *restrict in_gfs,
              REAL *restrict rhs_gfs) {
#include "set_CodeParameters-simd.h"
#pragma omp parallel for
  for (int i2 = NGHOSTS; i2 < Nxx_plus_2NGHOSTS2 - NGHOSTS; i2++) {
    for (int i1 = NGHOSTS; i1 < Nxx_plus_2NGHOSTS1 - NGHOSTS; i1++) {
      for (int i0 = NGHOSTS; i0 < Nxx_plus_2NGHOSTS0 - NGHOSTS; i0 += SIMD_WIDTH) {

        /*
         * NRPy-Generated GF Access/FD Code, Step 1 of 2:
         * Read gridfunction(s) from main memory and compute FD stencils as needed.
         */
        const REAL_SIMD_ARRAY uu_i2m2 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1, i2 - 2)]);
        const REAL_SIMD_ARRAY uu_i2m1 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1, i2 - 1)]);
        const REAL_SIMD_ARRAY uu_i1m2 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1 - 2, i2)]);
        const RE

## Step 8: Read the Diagnostics Source

Diagnostics write the errors used by convergence notebooks. The full file shows how a generated project turns field data into numbers that can be checked after a run. Look for the output filename, the error calculation, and the line that writes data.


In [9]:
print((PROJECT_DIR / "diagnostics.c").read_text(encoding="utf-8", errors="replace"))


#include "BHaH_defines.h"
#include "BHaH_function_prototypes.h"

/**
 * Diagnostics.
 */
void diagnostics(commondata_struct *restrict commondata, griddata_struct *restrict griddata) {
  const REAL currtime = commondata->time, currdt = commondata->dt, outevery = commondata->diagnostics_output_every;
  // Explanation of the if() below:
  // Step 1: round(currtime / outevery) gives the nearest integer n to the ratio currtime/outevery.
  // Step 2: Multiplying by outevery yields the nearest output time t_out = n * outevery.
  // Step 3: If fabs(t_out - currtime) < 0.5 * currdt, then currtime is as close to t_out as possible.
  if (fabs(round(currtime / outevery) * outevery - currtime) < 0.5 * currdt) {
    for (int grid = 0; grid < commondata->NUMGRIDS; grid++) {
      // Unpack griddata struct:
      const REAL *restrict y_n_gfs = griddata[grid].gridfuncs.y_n_gfs;
      REAL *restrict xx[3];
      for (int ww = 0; ww < 3; ww++)
        xx[ww] = griddata[grid].xx[ww];
      const params_st

The inspected files show how a BHaH project connects parameters, prototypes, generated C functions, and diagnostics. This structure is the target produced by the earlier C-function and wave notebooks.


## Learning Check

Before opening any full source file, predict which file should contain runtime settings and which should contain the wave update formula.


## Continue to ETLegacy Infrastructure
- [C Function Registration](../1-intro/c_function.ipynb)
- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb)
- [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)
